# Hallucination Checker Evaluation — Colab Reproduction

Reproduces the hallucination-detection evaluation from
`m3_implementation/test_result/hallucination_result/` using **the exact same
scripts** that run locally, so the results are identical by construction.

**Before running — build the upload bundle locally** (from the repo root on your machine):
```
venv\Scripts\python.exe m3_implementation\test_result\hallucination_result\make_colab_bundle.py
```
That creates `colab_bundle.zip` (checker code + captured cases + archived v1/v2 results + catalog CSV — no databases or models needed).

**What runs here:** Stage 2 (test-set construction, deterministic seed 42), Stage 3 (our checker + naive-NLI baseline; the Groq LLM judge is optional), Stage 4 (summary + figures).

**What cannot run here:** Stage 1 (live case capture) — it needs the local MongoDB/Redis/PostgreSQL/Qdrant stack, so the committed `captured_cases.jsonl` is used instead.

> Tip: use a GPU runtime (Runtime → Change runtime type → T4) — the DeBERTa NLI stages drop from ~20 min to ~3 min. Decisions are threshold-based, so CPU/GPU give the same verdicts.

In [ ]:
# 1 — Dependencies
!pip -q install sentence-transformers httpx python-dotenv matplotlib

In [ ]:
# 2 — Upload and extract colab_bundle.zip
import os, zipfile
from google.colab import files

up = files.upload()  # choose colab_bundle.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall('.')

EVAL_DIR = 'm3_implementation/test_result/hallucination_result'
assert os.path.exists(f'{EVAL_DIR}/captured_cases.jsonl'), 'bundle missing captured cases!'
assert os.path.exists('shared/main_data_set/sample_articles.csv'), 'bundle missing catalog CSV!'
print('bundle extracted OK')

In [ ]:
# 3 — Stage 2: build the labeled test set (deterministic, seed 42)
# Expect: 238 cases = 33 clean + 205 hallucinated
!python m3_implementation/test_result/hallucination_result/corrupt_cases.py

In [ ]:
# 4 — Optional: enable the Groq LLM-judge baseline
# Leave False to reproduce ours + naive-NLI only (no API key needed).
# The consolidated summary still shows the LLM judge via the archived v1 results.
RUN_LLM_JUDGE = False

if RUN_LLM_JUDGE:
    import getpass, os
    os.environ['GROQ_API_KEY'] = getpass.getpass('GROQ_API_KEY: ')
    os.environ['LLM_PROVIDER'] = 'groq'

In [ ]:
# 5 — Stage 3: run the detectors over all 238 cases
# Our checker (v3) + naive-NLI baseline; LLM judge if enabled above.
# Expected headline (ours, v3): P=1.000  R=0.951  F1=0.975  BalAcc=0.976
flags = '' if RUN_LLM_JUDGE else '--skip-llm'
!python m3_implementation/test_result/hallucination_result/run_detector_eval.py $flags

In [ ]:
# 6 — Stage 4: consolidated summary + figures
!python m3_implementation/test_result/hallucination_result/build_summary.py
!python m3_implementation/test_result/hallucination_result/make_figures.py

In [ ]:
# 7 — Show the headline table and figures
import json, os
from IPython.display import Image, display

EVAL_DIR = 'm3_implementation/test_result/hallucination_result'
S = json.load(open(f'{EVAL_DIR}/results_summary.json', encoding='utf-8'))

rows = [('Our checker (v3)', S['checker_versions']['v3']['metrics']),
        ('Our checker (v2)', S['checker_versions']['v2']['metrics']),
        ('Our checker (v1)', S['checker_versions']['v1']['metrics']),
        ('Naive NLI',        S['baselines']['naive_nli']['metrics']),
        ('LLM judge',        S['baselines']['llm_judge']['metrics'])]

print(f"{'system':<18} {'P':>7} {'R':>7} {'F1':>7} {'BalAcc':>7}")
print('-' * 50)
for name, m in rows:
    print(f"{name:<18} {m['precision']:>7.3f} {m['recall']:>7.3f} "
          f"{m['f1']:>7.3f} {m['balanced_accuracy']:>7.3f}")

figdir = f'{EVAL_DIR}/figures'
for f in sorted(os.listdir(figdir)):
    if f.endswith('.png'):
        display(Image(os.path.join(figdir, f)))

In [ ]:
# 8 — Optional: download the generated artifacts
from google.colab import files as gfiles
import shutil
shutil.make_archive('hallucination_eval_outputs', 'zip',
                    'm3_implementation/test_result/hallucination_result')
gfiles.download('hallucination_eval_outputs.zip')